In [15]:
# pip install ultralytics opencv-python-headless pillow numpy scikit-learn tqdm

In [16]:
# !pip install ultralytics opencv-python-headless pillow numpy scikit-learn tqdm
import os, glob, csv, math, json
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO

MODEL_PATH = r'C:\Users\juans\Documents\proarchitecg\version_2_docker\model_clasification_image_v2\runs\classify\person_cls\weights\best.pt'
IMG_DIR    = r"D:\historias\dev\imagenes_por_doc\LETRA A\ACTA N° 70\1\Abad Abad Rosa Elena"   # páginas ya separadas (.png/.jpg)
OUT_CSV    = r"C:\Users\juans\Documents\version_final_historias laborales\answer_model\inferencia_calidad.csv"

IMG_EXTS   = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
IMG_SIZE   = 1024  # 640-1280 según tu modelo y VRAM

# Umbrales de decisión (puedes retocarlos tras ver el CSV)
TAU_C = 0.55   # confianza mínima para aceptar
EPS_M = 0.15   # margen mínimo (conf_top1 - conf_top2)
TAU_E = 0.75   # entropía máxima aceptable (0..1)  (más alta = más incierto)

In [17]:
# -----------------------------------
# Utilidades de calidad (con calibración)
# -----------------------------------
def _entropy(gray):
    hist = cv2.calcHist([gray],[0],None,[256],[0,256]).ravel()
    p = hist / (hist.sum() + 1e-9)
    p = p[p>0]
    return float(-(p*np.log2(p)).sum())

def _exposure_ratio(gray, thr_low=10, thr_high=245):
    total = gray.size
    lows  = (gray < thr_low).sum()
    highs = (gray > thr_high).sum()
    return (lows + highs) / total

def collect_image_paths(root):
    return [str(p) for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXTS]

def calibrate_quality_ranges(sample_paths, p_low=5, p_high=95):
    lap_vals, ent_vals, exp_vals = [], [], []
    for p in sample_paths:
        img = cv2.imread(p)
        if img is None: continue
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        lap_vals.append(cv2.Laplacian(g, cv2.CV_64F).var())  # nitidez
        ent_vals.append(_entropy(g))                         # detalle/contraste
        exp_vals.append(_exposure_ratio(g))                  # sobre/sub exposición

    def P(arr, q, default):
        return float(np.percentile(arr, q)) if len(arr)>0 else default

    # Default de respaldo si la muestra fuera muy pequeña
    LAP_MIN, LAP_MAX = P(lap_vals, p_low, 50.0),  P(lap_vals, p_high, 800.0)
    ENT_MIN, ENT_MAX = P(ent_vals, p_low, 3.5),   P(ent_vals, p_high, 7.0)
    EXP_MIN, EXP_MAX = P(exp_vals, p_low, 0.00),  P(exp_vals, p_high, 0.30)

    return (LAP_MIN, LAP_MAX, ENT_MIN, ENT_MAX, EXP_MIN, EXP_MAX)

def _scale(x, a, b):
    return float(np.clip((x - a) / (b - a + 1e-9), 0, 1))

def build_quality_fn(ranges):
    LAP_MIN, LAP_MAX, ENT_MIN, ENT_MAX, EXP_MIN, EXP_MAX = ranges
    def quality_score(img_bgr):
        g  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        lap = cv2.Laplacian(g, cv2.CV_64F).var()
        ent = _entropy(g)
        exr = _exposure_ratio(g)

        lv = _scale(lap, LAP_MIN, LAP_MAX)        # nitidez (alto=mejor)
        en = _scale(ent, ENT_MIN, ENT_MAX)        # detalle (alto=mejor)
        ex = 1.0 - _scale(exr, EXP_MIN, EXP_MAX)  # exposición (alto=mejor)
        return 0.5*lv + 0.3*en + 0.2*ex
    return quality_score


In [18]:
# -----------------------------------
# Carga del modelo YOLO y nombres
# -----------------------------------
model = YOLO(MODEL_PATH)
names = model.names  # dict {id: label} o list
if isinstance(names, dict):
    CLASS_NAMES = [names[i] for i in range(max(names.keys())+1)]
else:
    CLASS_NAMES = list(names)

print("YOLO task:", getattr(model, "task", "unknown"))


YOLO task: classify


In [19]:
# =========================
# FUNCIÓN quality_score (0..1)
# =========================
def quality_score(img_bgr):
    g  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    lap = cv2.Laplacian(g, cv2.CV_64F).var()   # nitidez
    ent = _entropy(g)                          # detalle
    exr = _exposure_ratio(g)                   # exposición (peor si alto)

    lv = _scale(lap, LAP_MIN, LAP_MAX)         # más alto = más nítido
    en = _scale(ent, ENT_MIN, ENT_MAX)         # más alto = más detalle
    ex = 1.0 - _scale(exr, EXP_MIN, EXP_MAX)   # más alto = mejor exposición

    return 0.5*lv + 0.3*en + 0.2*ex


In [20]:
# -----------------------------------
# Inferencia con TTA + proba
# Soporta modelos de CLASIFICACIÓN y DETECCIÓN
# -----------------------------------
def softmax(x):
    x = np.array(x, dtype=np.float32)
    x = x - x.max()
    e = np.exp(x)
    return e / (e.sum() + 1e-9)

def yolo_predict_proba(img_bgr, imgsz=IMG_SIZE):
    task = getattr(model, "task", None)
    # Tres vistas: original, flip H, gamma suave
    img_o = img_bgr
    img_f = cv2.flip(img_bgr, 1)
    img_g = np.clip(((img_bgr/255.0)**0.8)*255, 0, 255).astype(np.uint8)

    views = [img_o, img_f, img_g]
    logits_accum = None
    C = len(CLASS_NAMES)

    for v in views:
        r = model.predict(source=v, imgsz=imgsz, conf=0.001, iou=0.5, verbose=False)[0]

        # --- CLASIFICACIÓN ---
        if (hasattr(r, "probs") and (r.probs is not None)) or task == "classify":
            p = r.probs.data.float().cpu().numpy()  # shape (C,)
            logit = np.log(p + 1e-9)
            logits_accum = logit if logits_accum is None else (logits_accum + logit)
            continue

        # --- DETECCIÓN / SEGMENTACIÓN ---
        if hasattr(r, "boxes") and (r.boxes is not None) and (len(r.boxes) > 0):
            vec = np.full(C, -np.inf, dtype=np.float32)  # -inf = sin evidencia para esa clase
            for b in r.boxes:
                cls = int(b.cls.item())
                conf = float(b.conf.item())
                vec[cls] = max(vec[cls], np.log(conf + 1e-9))  # max log-conf por clase
            logits_accum = vec if logits_accum is None else np.maximum(logits_accum, vec)
            continue

    if logits_accum is None:
        return {"no_evidence": True, "reason": "no_preds_any_view"}

    probs = np.exp(logits_accum - logits_accum.max())
    probs = probs / (probs.sum() + 1e-9)

    order = np.argsort(-probs)
    top1 = int(order[0])
    top2 = int(order[1]) if len(order) > 1 else top1
    conf1 = float(probs[top1])
    conf2 = float(probs[top2])
    margin = conf1 - conf2
    # entropía normalizada (0..1 aprox.)
    entropy = float(-(probs * np.log2(probs + 1e-9)).sum() / math.log2(len(probs)))

    return {
        "proba": probs,
        "top1": top1, "conf1": conf1,
        "top2": top2, "conf2": conf2,
        "margin": margin, "entropy": entropy
    }


In [21]:
# -----------------------------------
# Calibración de calidad + fijar TAU_Q
# -----------------------------------
all_imgs = collect_image_paths(IMG_DIR)
if len(all_imgs) == 0:
    raise ValueError(f"No se encontraron imágenes en: {IMG_DIR}")

SAMPLE_N = min(300, len(all_imgs))
ranges = calibrate_quality_ranges(all_imgs[:SAMPLE_N])
quality_score = build_quality_fn(ranges)

# Calcular percentiles de quality_score para sugerir TAU_Q
qs = []
for p in all_imgs[:SAMPLE_N]:
    img = cv2.imread(p); 
    if img is None: continue
    qs.append(quality_score(img))

qs = np.array(qs) if len(qs)>0 else np.array([0.5])
pcts = np.percentile(qs, [5,10,15,25,50,75,90,95]).tolist()
# Umbral sugerido: percentil 15 (ajústalo si quieres)
TAU_Q = float(pcts[2])

print("Rangos calidad (LAP, ENT, EXP):", ranges)
print("Quality percentiles (5..95):", [round(x,3) for x in pcts])
print("TAU_Q (sugerido p15):", round(TAU_Q,3))


Rangos calidad (LAP, ENT, EXP): (1573.5742086060204, 14198.078612958558, 0.35272846668958663, 1.159220394492149, 0.9619532088987462, 0.9986480091922688)
Quality percentiles (5..95): [0.085, 0.118, 0.146, 0.147, 0.165, 0.483, 0.705, 0.833]
TAU_Q (sugerido p15): 0.146


In [22]:
# -----------------------------------
# Decisión final por página
# -----------------------------------
def decide(quality, conf1, margin, entropy, no_evidence=False):
    if no_evidence:
        return "unknown_no_preds"
    if quality < TAU_Q:
        return "unknown_low_quality"
    if (conf1 < TAU_C) or (margin < EPS_M) or (entropy > TAU_E):
        return "unknown_uncertain"
    return "accept"

# -----------------------------------
# Bucle principal + CSV
# -----------------------------------
rows = []
for p in tqdm(all_imgs, desc="Inferencia"):
    img = cv2.imread(p)
    if img is None:
        rows.append(dict(
            page_path=p, quality_score=None,
            label_top1=None, conf_top1=None,
            label_top2=None, conf_top2=None,
            margin=None, entropy=None,
            decision="unknown_io"
        ))
        continue

    q = float(quality_score(img))
    pred = yolo_predict_proba(img, imgsz=IMG_SIZE)

    if pred.get("no_evidence", False):
        rows.append(dict(
            page_path=p, quality_score=round(q,3),
            label_top1=None, conf_top1=None,
            label_top2=None, conf_top2=None,
            margin=None, entropy=None,
            decision=decide(q, 0.0, 0.0, 1.0, no_evidence=True)
        ))
        continue

    label1 = CLASS_NAMES[pred["top1"]]
    label2 = CLASS_NAMES[pred["top2"]]
    conf1  = float(pred["conf1"])
    conf2  = float(pred["conf2"])
    margin = float(pred["margin"])
    entro  = float(pred["entropy"])

    rows.append(dict(
        page_path=p, quality_score=round(q,3),
        label_top1=label1, conf_top1=round(conf1,3),
        label_top2=label2, conf_top2=round(conf2,3),
        margin=round(margin,3), entropy=round(entro,3),
        decision=decide(q, conf1, margin, entro, no_evidence=False)
    ))

# Guardar CSV
Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

# Resumen
from collections import Counter
cnt = Counter([r["decision"] for r in rows])
print(f"\nCSV guardado en: {OUT_CSV}")
print("Decisiones:", dict(cnt))

Inferencia: 100%|██████████| 8/8 [00:05<00:00,  1.60it/s]


CSV guardado en: C:\Users\juans\Documents\version_final_historias laborales\answer_model\inferencia_calidad.csv
Decisiones: {'unknown_low_quality': 2, 'accept': 6}
